# Fine-Tuning Laya on FAULTLINE OVERCONSTRAINED_SEARCH_LOOP (Kaggle GPU T4 ×2 DDP)

This notebook fine-tunes the English checkpoint of **Laya** (`convaiinnovations/laya`, ModernBERT-large 421M params) on the FAULTLINE training split (`train.jsonl`, 101 cases) and fits temperature calibration on the validation split (`dev.jsonl`, 39 cases).

Adapted directly from Laya's official fine-tuning notebook (`laya_finetune_typed_decisions_2xT4_kaggle.ipynb`).

---

### ⚠️ Kaggle Notebook Settings:
* **Accelerator:** Select **`GPU T4 x2`** in Notebook options
* **Internet:** Set to **`On`** (for downloading base weights and dependencies)
* **Input Dataset:** Attach dataset containing `train.jsonl` and `dev.jsonl` (e.g., `/kaggle/input/faultline-p17-data/`)
* **Output:** Final model and calibration config are saved to `/kaggle/working/laya_finetuned_overconstrained`

## 1. Environment & Dual T4 GPU Verification

In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")

## 2. Install Dependencies

In [ ]:
!pip install -q -U "laya>=0.3.5" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)

## 3. Preprocess Train and Dev Data

In [ ]:
import os, json, glob, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
print(f"Fetching base model config and tokenizer from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

# Locate train.jsonl and dev.jsonl
def find_data_file(filename):
    candidates = [
        f"/kaggle/input/{filename}",
        f"/kaggle/input/**/{filename}",
        f"./{filename}",
        f"../{filename}",
    ]
    for c in candidates:
        matches = glob.glob(c, recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Please attach dataset to Kaggle notebook.")

train_path = find_data_file("train.jsonl")
dev_path = find_data_file("dev.jsonl")
print(f"Found train: {train_path}")
print(f"Found dev  : {dev_path}")

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[t],
        "target": target,
        "label": label,
    }

def process_file(filepath):
    items = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            state = row["state"]
            questions = row.get("questions") or {"detected": row["question"]}
            gold = row.get("gold") or {"detected": {"type": "noul", "probabilities": {"false": 1.0 - row["label"], "true": float(row["label"])}}}
            for qid, q in questions.items():
                if qid in gold:
                    it = build_training_item(state, q, gold[qid])
                    if it:
                        items.append(it)
    return items

train_items = process_file(train_path)
dev_items = process_file(dev_path)

print(f"Preprocessed {len(train_items)} train items, {len(dev_items)} dev items.")
torch.save(train_items, "/kaggle/working/train_items.pt")
torch.save(dev_items, "/kaggle/working/dev_items.pt")
print("Saved preprocessed tensors to /kaggle/working/")

## 4. DDP Training Script (`train_ddp.py`)
Writes the multi-GPU distributed RLCD fine-tuning script with proper scoring rule reward and calibration on held-out dev.

In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 5:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    train_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    dev_items = torch.load("/kaggle/working/dev_items.pt", weights_only=False)
    my_items = train_items[rank::world_size]
    
    EPOCHS = 6
    MICRO_BATCH = 4
    GRAD_ACCUM = 4
    GROUP_SIZE = 4
    LR_ENCODER = 2.0e-5
    LR_HEAD = 1.0e-4
    SIGMA_START = 0.4
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP fine-tuning: {len(train_items)} train items | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")
        dist.barrier()

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 using DEV items
    if rank == 0:
        print("\nFitting post-training calibration temperatures on dev split...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(dev_items), 16):
                c_chunk = dev_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.0, 1.0, 1.0]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
            
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-finetuned-overconstrained"
        cfg["temperature"] = fitted_temps
        cfg.pop("temperature_by_options", None)
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Dual-GPU Fine-Tuning

In [ ]:
OUTPUT_DIR = "/kaggle/working/laya_finetuned_overconstrained"
cmd = f"torchrun --standalone --nproc_per_node=2 /kaggle/working/train_ddp.py {model_dir} {OUTPUT_DIR}"
print("Launching training:", cmd)
!{cmd}

## 6. Validate Saved Checkpoint on Dev Split

In [ ]:
import laya, json

agent = laya.load("/kaggle/working/laya_finetuned_overconstrained")
print("Loaded fine-tuned agent successfully:", agent)

dev_rows = [json.loads(line) for line in open(dev_path) if line.strip()]

pos_probs = []
neg_probs = []

for r in dev_rows:
    q = r.get("questions") or {"detected": r["question"]}
    res = agent.predict(r["state"], q)
    p = float(res["answers"]["detected"]["noul"])
    sid = r.get("scenario_id", "")
    if r["label"] == 1:
        pos_probs.append((sid, p))
    else:
        neg_probs.append((sid, p))

print("\n--- Dev Evaluation (noul probabilities) ---")
print(f"Positives (label=1, n={len(pos_probs)}):")
for sid, p in pos_probs:
    print(f"  {sid}: p = {p:.4f}")

print(f"\nNegatives (label=0, n={len(neg_probs)}):")
for sid, p in neg_probs:
    print(f"  {sid}: p = {p:.4f}")

lowest_pos_p = min(p for _, p in pos_probs) if pos_probs else 0.5
tp = sum(1 for _, p in pos_probs if p >= lowest_pos_p)
fn = sum(1 for _, p in pos_probs if p < lowest_pos_p)
tn = sum(1 for _, p in neg_probs if p < lowest_pos_p)
fp = sum(1 for _, p in neg_probs if p >= lowest_pos_p)

tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print(f"\nLowest positive probability on dev: {lowest_pos_p:.4f}")
print(f"Metrics at lowest positive probability (tau = {lowest_pos_p:.4f}):")
print(f"  TP: {tp}, FP: {fp}, TN: {tn}, FN: {fn}")
print(f"  TPR: {tpr:.4f} ({tp}/{tp+fn}), TNR: {tnr:.4f} ({tn}/{tn+fp})")
print("Ready to download checkpoint files from Kaggle output directory.")